# Bài 05: Các Chỉ số Đánh giá cho Dữ liệu Mất cân bằng (Metrics for Imbalanced Data)

**Mục tiêu bài học:**
- Hiểu được hạn chế của các chỉ số tiêu chuẩn khi áp dụng trên dữ liệu mất cân bằng.
- Học và áp dụng các chỉ số đánh giá chuyên biệt: Balanced Accuracy, G-Mean, và F-beta Score.
- Biết khi nào nên sử dụng mỗi loại chỉ số để phản ánh đúng hiệu suất của mô hình.

## 1. Tại sao cần các chỉ số mới?

Ở bài trước, chúng ta đã thấy rõ rằng Accuracy trở nên vô nghĩa và ROC AUC có thể gây hiểu lầm trên dữ liệu mất cân bằng. Mô hình có thể đạt Accuracy cao chỉ bằng cách dự đoán tất cả là lớp đa số, trong khi Recall của lớp thiểu số (thứ chúng ta thực sự quan tâm) lại cực kỳ thấp.

Các chỉ số như Precision, Recall, và F1-score (tính trên từng lớp) cung cấp một cái nhìn chi tiết hơn, nhưng chúng ta thường cần một con số **tổng hợp duy nhất** để so sánh nhanh các mô hình với nhau. Tuy nhiên, việc lấy trung bình đơn giản (macro-average) của các chỉ số này có thể không phản ánh đúng tầm quan trọng của mỗi lớp.

Do đó, cộng đồng khoa học dữ liệu đã phát triển các chỉ số được thiết kế đặc biệt để giải quyết vấn đề này. Chúng tìm cách cân bằng hiệu suất trên cả lớp đa số và lớp thiểu số, đưa ra một bức tranh toàn diện và đáng tin cậy hơn.

## 2. Chuẩn bị Dữ liệu

Chúng ta sẽ tiếp tục sử dụng bộ dữ liệu mất cân bằng đã tạo ở bài trước để minh họa.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Tạo bộ dữ liệu mất cân bằng (95% lớp 0, 5% lớp 1)
X, y = make_classification(n_samples=10000,
                           n_features=10,
                           n_informative=5,
                           n_redundant=0,
                           n_classes=2,
                           weights=[0.95, 0.05],
                           flip_y=0.05, # Thêm nhiễu
                           random_state=42)

# Chia dữ liệu
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Huấn luyện mô hình Logistic Regression đơn giản
model = LogisticRegression(solver='liblinear', random_state=42)
model.fit(X_train, y_train)

# Đưa ra dự đoán
y_pred = model.predict(X_test)

# In lại classification report để nhớ lại vấn đề
print("Classification Report trên dữ liệu mất cân bằng:")
print(classification_report(y_test, y_pred))

Nhìn vào report, chúng ta thấy Accuracy cao (95%) nhưng Recall của lớp 1 rất thấp (chỉ 0.21). Đây chính là vấn đề chúng ta cần giải quyết.

## 3. Các chỉ số chuyên biệt

### 3.1. Balanced Accuracy

**Ý tưởng:** Balanced Accuracy là trung bình cộng của Recall (hay còn gọi là Sensitivity) của mỗi lớp. Nó tránh việc bị ảnh hưởng bởi kích thước của các lớp.

**Công thức:**
$$ \text{Balanced Accuracy} = \frac{\text{Sensitivity} + \text{Specificity}}{2} = \frac{\text{Recall}_{	ext{lớp 1}} + \text{Recall}_{	ext{lớp 0}}}{2} $$

Trong đó:
- **Sensitivity (Độ nhạy)** chính là Recall của lớp dương tính (lớp 1): $ \frac{TP}{TP+FN} $
- **Specificity (Độ đặc hiệu)** chính là Recall của lớp âm tính (lớp 0): $ \frac{TN}{TN+FP} $

**Diễn giải:**
- Nếu mô hình dự đoán tốt cả hai lớp, Balanced Accuracy sẽ cao.
- Nếu mô hình chỉ dự đoán tốt lớp đa số và bỏ qua lớp thiểu số, Balanced Accuracy sẽ thấp (gần 0.5 cho mô hình ngẫu nhiên), phơi bày sự yếu kém của mô hình.
- Nó trừng phạt nặng các mô hình chỉ học vẹt lớp đa số.

In [ ]:
from sklearn.metrics import balanced_accuracy_score

# Tính toán các chỉ số
acc = accuracy_score(y_test, y_pred)
balanced_acc = balanced_accuracy_score(y_test, y_pred)

print(f"Standard Accuracy: {acc:.4f}")
print(f"Balanced Accuracy: {balanced_acc:.4f}")

# Tính thủ công để hiểu rõ hơn
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
sensitivity = tp / (tp + fn) # Recall lớp 1
specificity = tn / (tn + fp) # Recall lớp 0
manual_balanced_acc = (sensitivity + specificity) / 2

print(f"\n--- Tính toán thủ công ---")
print(f"Recall lớp 1 (Sensitivity): {sensitivity:.4f}")
print(f"Recall lớp 0 (Specificity): {specificity:.4f}")
print(f"Manual Balanced Accuracy: {manual_balanced_acc:.4f}")

**Nhận xét:**
Rõ ràng, trong khi Accuracy là **0.9513**, Balanced Accuracy chỉ là **0.5993**. Con số này phản ánh đúng hơn nhiều về hiệu suất thực tế của mô hình: nó làm rất tốt trên lớp 0 (Recall ~0.99) nhưng rất tệ trên lớp 1 (Recall ~0.21), và Balanced Accuracy đã "vạch trần" sự mất cân bằng này.

### 3.2. G-Mean (Geometric Mean)

**Ý tưởng:** G-Mean là trung bình nhân của Sensitivity và Specificity. Đặc tính của trung bình nhân là nó sẽ bị kéo về phía giá trị nhỏ hơn. Do đó, G-Mean chỉ cao khi cả hai chỉ số thành phần đều cao.

**Công thức:**
$$ \text{G-Mean} = \sqrt{\text{Sensitivity} \times \text{Specificity}} = \sqrt{\text{Recall}_{	ext{lớp 1}} \times \text{Recall}_{	ext{lớp 0}}} $$

**Diễn giải:**
- G-Mean đặc biệt nhạy cảm với hiệu suất kém trên bất kỳ lớp nào.
- Nếu mô hình có Recall lớp 1 rất thấp (gần 0), G-Mean cũng sẽ bị kéo xuống gần 0, ngay cả khi Recall lớp 0 rất cao.
- Đây là một chỉ số rất "nghiêm khắc", phù hợp cho các bài toán mà việc bỏ sót lớp thiểu số là không thể chấp nhận được.

In [ ]:
# Scikit-learn không có sẵn G-Mean, nhưng chúng ta có thể dễ dàng tính toán
# Hoặc sử dụng thư viện imbalanced-learn
from imblearn.metrics import geometric_mean_score

g_mean = geometric_mean_score(y_test, y_pred)

print(f"G-Mean Score: {g_mean:.4f}")

# Tính thủ công
manual_g_mean = np.sqrt(sensitivity * specificity)
print(f"Manual G-Mean: {manual_g_mean:.4f}")

**Nhận xét:**
G-Mean là **0.4547**, thậm chí còn thấp hơn Balanced Accuracy. Điều này là do `sensitivity` (0.21) rất thấp, và phép nhân đã "trừng phạt" kết quả nặng nề hơn so với phép cộng của Balanced Accuracy. G-Mean cho thấy một cách rõ ràng rằng mô hình đang hoạt động rất kém trong việc cân bằng giữa hai lớp.

### 3.3. F-beta Score

**Ý tưởng:** Chúng ta đã quen thuộc với F1-score, là trung bình điều hòa của Precision và Recall, coi trọng hai chỉ số này như nhau. Tuy nhiên, trong nhiều bài toán, chúng ta có thể muốn ưu tiên một trong hai hơn.

- **Ưu tiên Recall:** Trong y tế, việc bỏ sót một bệnh nhân (False Negative) nguy hiểm hơn nhiều so với việc chẩn đoán nhầm một người khỏe mạnh (False Positive). Chúng ta muốn tối đa hóa Recall.
- **Ưu tiên Precision:** Trong việc đề xuất video cho người dùng, việc đề xuất một video không liên quan (False Positive) sẽ làm giảm trải nghiệm người dùng. Chúng ta muốn tối đa hóa Precision.

F-beta score cho phép chúng ta điều chỉnh sự cân bằng này thông qua tham số `beta`.

**Công thức:**
$$ F_\beta = (1 + \beta^2) \cdot \frac{\text{Precision} \cdot \text{Recall}}{(\beta^2 \cdot \text{Precision}) + \text{Recall}} $$

**Diễn giải:**
- **`beta = 1`**: Chính là F1-score, Precision và Recall quan trọng như nhau.
- **`beta < 1` (ví dụ: 0.5)**: Ưu tiên Precision hơn Recall.
- **`beta > 1` (ví dụ: 2)**: Ưu tiên Recall hơn Precision. `beta=2` có nghĩa là Recall được coi trọng gấp đôi so với Precision.

Trong các bài toán mất cân bằng, chúng ta thường quan tâm đến việc tìm ra lớp thiểu số, vì vậy **F2-score (`beta=2`)** là một lựa chọn phổ biến.

In [ ]:
from sklearn.metrics import fbeta_score, f1_score

# Tính các F-score cho lớp thiểu số (pos_label=1)
f1 = f1_score(y_test, y_pred, pos_label=1)
f0_5 = fbeta_score(y_test, y_pred, beta=0.5, pos_label=1) # Ưu tiên Precision
f2 = fbeta_score(y_test, y_pred, beta=2, pos_label=1)   # Ưu tiên Recall

print(f"F-0.5 Score (ưu tiên Precision): {f0_5:.4f}")
print(f"F-1 Score (cân bằng):          {f1:.4f}")
print(f"F-2 Score (ưu tiên Recall):    {f2:.4f}")

**Nhận xét:**
- **F1-score** là 0.30, một con số thấp, phản ánh sự cân bằng kém giữa Precision (0.58) và Recall (0.21).
- **F0.5-score** cao hơn (0.44), vì nó được kéo lên bởi Precision tương đối ổn.
- **F2-score** thấp nhất (0.24), vì nó bị kéo xuống bởi Recall rất tệ. Trong bài toán này, F2-score là chỉ số trung thực nhất nếu mục tiêu của chúng ta là tìm ra càng nhiều trường hợp lớp 1 càng tốt.

Việc chọn `beta` phụ thuộc hoàn toàn vào mục tiêu của bài toán.

## 4. Tổng kết và Khi nào dùng gì?

| Chỉ số             | Công thức                                       | Ưu điểm                                                                      | Nhược điểm                                                              | Khi nào nên dùng?                                                                                             |
|--------------------|-------------------------------------------------|------------------------------------------------------------------------------|-------------------------------------------------------------------------|---------------------------------------------------------------------------------------------------------------|
| **Accuracy**       | `(TP+TN)/(TP+TN+FP+FN)`                         | Dễ hiểu, dễ tính.                                                            | **Vô dụng** trên dữ liệu mất cân bằng.                                  | Chỉ dùng khi các lớp gần như cân bằng và chi phí dự đoán sai các lớp là như nhau.                              |
| **Balanced Acc.**  | `(Recall_0 + Recall_1)/2`                       | Cân bằng hiệu suất hai lớp, không bị ảnh hưởng bởi kích thước lớp.           | Có thể không phân biệt được mô hình có (Recall_0=0.9, Recall_1=0.1) với (Recall_0=0.5, Recall_1=0.5). | Một chỉ số tổng quan tốt để thay thế Accuracy trong hầu hết các bài toán mất cân bằng.                          |
| **G-Mean**         | `sqrt(Recall_0 * Recall_1)`                     | **Rất nghiêm khắc**, trừng phạt nặng nếu hiệu suất trên bất kỳ lớp nào kém.   | Có thể quá nhạy cảm, một sự sụt giảm nhỏ ở một lớp có thể làm giảm mạnh G-Mean. | Khi việc đạt được hiệu suất tốt trên **cả hai lớp** là cực kỳ quan trọng và không thể đánh đổi.                 |
| **F-beta Score**   | `(1+β²) * (P*R) / (β²*P + R)`                   | **Linh hoạt**, cho phép tùy chỉnh tầm quan trọng của Precision và Recall.     | Cần phải xác định rõ `beta` dựa trên yêu cầu bài toán.                   | Khi có sự ưu tiên rõ ràng giữa Precision và Recall (ví dụ: F2 cho y tế, F0.5 cho hệ thống gợi ý).             |
| **PR AUC**         | Diện tích dưới đường Precision-Recall          | Tập trung vào hiệu suất trên lớp thiểu số, ít bị ảnh hưởng bởi lớp đa số.      | Không phải là một ngưỡng cụ thể, khó diễn giải trực tiếp.                | Để so sánh tổng thể các mô hình khác nhau về khả năng phân biệt lớp thiểu số trên mọi ngưỡng.                 |

**Lời khuyên:**
Không có một chỉ số nào là hoàn hảo. Trong thực tế, bạn nên:
1.  Luôn bắt đầu bằng `classification_report` để xem chi tiết Precision, Recall, F1-score cho từng lớp.
2.  Chọn một chỉ số tổng hợp chính (ví dụ: **Balanced Accuracy** hoặc **F2-score**) để tối ưu hóa và so sánh mô hình, dựa trên mục tiêu kinh doanh.
3.  Sử dụng **PR Curve** và **PR AUC** để có cái nhìn tổng quan về khả năng phân loại của mô hình trên các ngưỡng khác nhau.

## 5. Kết luận

Chúng ta đã trang bị thêm những công cụ đánh giá mạnh mẽ và phù hợp hơn cho các bài toán dữ liệu mất cân bằng. Việc lựa chọn đúng chỉ số không chỉ giúp chúng ta đo lường chính xác hơn mà còn định hướng cho việc cải thiện mô hình sau này.

Tuy nhiên, việc đo lường tốt hơn mới chỉ là một nửa của câu chuyện. Làm thế nào để chúng ta thực sự **cải thiện** các chỉ số này? Trong bài học tiếp theo, chúng ta sẽ khám phá nhóm kỹ thuật đầu tiên để xử lý dữ liệu mất cân bằng: **Resampling (Lấy mẫu lại)**, bao gồm Undersampling và Oversampling.